In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from flask_cors import CORS

In [2]:
from flask import Flask, request, jsonify

## Import Pickle

In [3]:
import pickle

In [4]:
file = open('rain_model.pkl', 'rb')
rain_model = pickle.load(file)
file.close()

In [5]:
file = open('wind_model.pkl', 'rb')
wind_model = pickle.load(file)
file.close()

In [6]:
file = open('temp_model.pkl', 'rb')
temp_model = pickle.load(file)
file.close()

In [7]:
file = open('daily_humidity.pkl', 'rb')
daily_humidity = pickle.load(file)
file.close()

file = open('daily_pressure.pkl', 'rb')
daily_pressure = pickle.load(file)
file.close()

file = open('daily_temp.pkl', 'rb')
daily_temp = pickle.load(file)
file.close()

file = open('daily_wind_speed.pkl', 'rb')
daily_wind_speed = pickle.load(file)
file.close()

file = open('daily_longwave.pkl', 'rb')
daily_longwave = pickle.load(file)
file.close()

file = open('daily_shortwave.pkl', 'rb')
daily_shortwave = pickle.load(file)
file.close()

file = open('daily_precip.pkl'  , 'rb')
daily_precip = pickle.load(file)
file.close()

file = open('daily_wind_speed.pkl', 'rb')
daily_wind_speed = pickle.load(file)
file.close()




In [8]:
#Define a function that takes two dates and produces an array of dates between them
def split_dates(start, end):
    dates = pd.date_range(start, end)
    return dates

## Get/Post

In [9]:
app = Flask(__name__)
CORS(app, origins=["http://localhost:5173"])

### Here

In [10]:
def get_prediction(date, latitude, longitude):
    
        month = date.month
        day = date.day
        year = date.year
        hour = date.hour
        
      
        rain_df = {        
        'Humidity':[daily_humidity['Humidity'][month*30 +day]],
        'Pressure (Pa)': [daily_pressure['Pressure (Pa)'][month*30 +day]],
        'Downward Shortwave Radiation (W/m^2)' : [daily_shortwave['Downward Shortwave Radiation (W/m^2)'][month*30 + day]],
        'Downward Longwave Radiation (W/m^2)' : [daily_longwave['Downward Longwave Radiation (W/m^2)'][month*30 + day]],
        'Year':[year],
        'Month':[month],
        'Day':[day],
        'Hour':[hour],
        'Longitude':[longitude],
        'Latitude':[latitude]}
    
        wind_df = {        
        'Surface Air Pressure (Pa)' : [daily_pressure['Pressure (Pa)'][month * 30 + day]],
        'Surface Air Temp (K)': [daily_temp['Surface Air Temp (K)'][month * 30 + day]],
        'Year':[year],
        'Month':[month],
        'Day':[day],
        'Hour':[hour],
        'Longitude':[longitude],
        'Latitude':[latitude]}
   
        temp_df = {
        'Surface Wind Speed (m/s)':[daily_wind_speed['Surface Wind Speed (m/s)'][month * 30 + day]],
        'Humidity (g/kg)': [daily_humidity['Humidity'][month * 30 + day]],
        'Year':[year],
        'Month':[month],
        'Day':[day],
        'Hour':[hour],
        'Longitude':[longitude],
        'Latitude':[latitude] }

        rain_df1 = pd.DataFrame(rain_df)
        wind_df = pd.DataFrame(wind_df)
        temp_df = pd.DataFrame(temp_df)

        #convert rain into mm/hr
        rain = rain_model.predict(rain_df1)[0]
        rain = rain * 3600
        #convert wind into km/hr
        wind = wind_model.predict(wind_df)[0]
        wind = wind * 3.6
        #convert temperature into Celsius
        temp = temp_model.predict(temp_df)[0]
        temp = temp - 273.15



        prediction = {'Date': date, 'Rain': rain, 'Wind':wind, 'Temp':temp}
        return prediction

## Method 1 Range of Date

In [11]:
@app.route("/request", methods=["POST"])
def predict2():

    data = request.get_json()

    latitude = data['Latitude']
    longitude = data['Longitude']
    
    start_date = data['Start_Date']
    end_date = data['End_Date']
    date_list = split_dates(start_date, end_date)

    pred_list = []
    for each in date_list:
        prediction = get_prediction(each, latitude, longitude)
        pred_list.append(prediction)
    
    print(pred_list)
    return jsonify(pred_list)

In [ ]:
if __name__ == "__main__":
    app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit


127.0.0.1 - - [05/Oct/2025 21:12:03] "OPTIONS /request HTTP/1.1" 200 -
[2025-10-05 21:12:04,274] ERROR in app: Exception on /request [POST]
Traceback (most recent call last):
  File "c:\Users\Manjeet\anaconda3\Lib\site-packages\pandas\core\indexes\range.py", line 413, in get_loc
    return self._range.index(new_key)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: 366 is not in range

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\Users\Manjeet\anaconda3\Lib\site-packages\flask\app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Manjeet\anaconda3\Lib\site-packages\flask\app.py", line 919, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Manjeet\anaconda3\Lib\site-packages\flask_cors\extension.py", line 176, in wrapped_function
    return cors_after_request(app.mak

[{'Date': Timestamp('2026-01-01 05:00:00'), 'Rain': 0.04230306434427457, 'Wind': 15.283209512757436, 'Temp': 0.3329814130840987}, {'Date': Timestamp('2026-01-02 05:00:00'), 'Rain': 0.03891702846685343, 'Wind': 15.025202775493382, 'Temp': 0.06614217359805252}, {'Date': Timestamp('2026-01-03 05:00:00'), 'Rain': 0.038503749204508456, 'Wind': 15.113873028415947, 'Temp': 0.433931311418462}, {'Date': Timestamp('2026-01-04 05:00:00'), 'Rain': 0.04185921987181454, 'Wind': 14.954852478444947, 'Temp': 0.20202468938856555}, {'Date': Timestamp('2026-01-05 05:00:00'), 'Rain': 0.03266480372786219, 'Wind': 15.08478642415234, 'Temp': -0.10286786070969356}, {'Date': Timestamp('2026-01-06 05:00:00'), 'Rain': 0.037614145601827735, 'Wind': 14.850057022168874, 'Temp': -0.18362597922123314}, {'Date': Timestamp('2026-01-07 05:00:00'), 'Rain': 0.04807621457589084, 'Wind': 15.187756276631916, 'Temp': 0.2078453447170432}, {'Date': Timestamp('2026-01-08 05:00:00'), 'Rain': 0.029642644800450528, 'Wind': 14.991125

127.0.0.1 - - [05/Oct/2025 21:14:02] "OPTIONS /request HTTP/1.1" 200 -
127.0.0.1 - - [05/Oct/2025 21:14:02] "POST /request HTTP/1.1" 200 -


...

[{'Date': Timestamp('2025-10-07 04:00:00'), 'Rain': 0.07827994677948155, 'Wind': 12.246491648729757, 'Temp': 7.786304011739844}, {'Date': Timestamp('2025-10-08 04:00:00'), 'Rain': 0.07920052993593281, 'Wind': 12.003994867219882, 'Temp': 7.82302347901026}, {'Date': Timestamp('2025-10-09 04:00:00'), 'Rain': 0.09680010291842715, 'Wind': 12.484730475665282, 'Temp': 9.411594238711018}, {'Date': Timestamp('2025-10-10 04:00:00'), 'Rain': 0.08698532423671267, 'Wind': 12.995042928212621, 'Temp': 8.8667209002964}, {'Date': Timestamp('2025-10-11 04:00:00'), 'Rain': 0.0851187374441194, 'Wind': 12.759016189963274, 'Temp': 7.63784553611265}, {'Date': Timestamp('2025-10-12 04:00:00'), 'Rain': 0.06590804537588729, 'Wind': 12.383781136142472, 'Temp': 7.340944892091386}, {'Date': Timestamp('2025-10-13 04:00:00'), 'Rain': 0.08309170322580385, 'Wind': 12.69507389423785, 'Temp': 7.592399710489985}, {'Date': Timestamp('2025-10-14 04:00:00'), 'Rain': 0.0849495321935558, 'Wind': 13.38419967620381, 'Temp': 7.6

127.0.0.1 - - [05/Oct/2025 21:25:36] "OPTIONS /request HTTP/1.1" 200 -
127.0.0.1 - - [05/Oct/2025 21:25:37] "POST /request HTTP/1.1" 200 -


[{'Date': Timestamp('2025-10-06 04:00:00'), 'Rain': 0.07586886057149693, 'Wind': 12.607239873902307, 'Temp': 7.688137644015853}, {'Date': Timestamp('2025-10-07 04:00:00'), 'Rain': 0.07827994677948155, 'Wind': 12.246491648729757, 'Temp': 7.779498607924609}, {'Date': Timestamp('2025-10-08 04:00:00'), 'Rain': 0.07920052993593281, 'Wind': 12.003994867219882, 'Temp': 7.8162180751950245}, {'Date': Timestamp('2025-10-09 04:00:00'), 'Rain': 0.09680010291842715, 'Wind': 12.484730475665282, 'Temp': 9.404788834895783}, {'Date': Timestamp('2025-10-10 04:00:00'), 'Rain': 0.08698532423671267, 'Wind': 12.995042928212621, 'Temp': 8.859915496481165}, {'Date': Timestamp('2025-10-11 04:00:00'), 'Rain': 0.0851187374441194, 'Wind': 12.759016189963274, 'Temp': 7.631040132297358}, {'Date': Timestamp('2025-10-12 04:00:00'), 'Rain': 0.06590804537588729, 'Wind': 12.383781136142472, 'Temp': 7.334139488276151}, {'Date': Timestamp('2025-10-13 04:00:00'), 'Rain': 0.08309170322580385, 'Wind': 12.69507389423785, 'Tem

127.0.0.1 - - [05/Oct/2025 21:25:49] "OPTIONS /request HTTP/1.1" 200 -
127.0.0.1 - - [05/Oct/2025 21:25:50] "POST /request HTTP/1.1" 200 -


[{'Date': Timestamp('2025-09-07 04:00:00'), 'Rain': 0.1242494394390256, 'Wind': 11.028403725072621, 'Temp': 15.059838393002565}, {'Date': Timestamp('2025-09-08 04:00:00'), 'Rain': 0.09650929436323304, 'Wind': 10.694721049734483, 'Temp': 12.898760078460725}, {'Date': Timestamp('2025-09-09 04:00:00'), 'Rain': 0.11212069948835748, 'Wind': 11.048746661395944, 'Temp': 13.936328591032066}, {'Date': Timestamp('2025-09-10 04:00:00'), 'Rain': 0.11651202642005802, 'Wind': 11.719422892428993, 'Temp': 13.23668059406964}, {'Date': Timestamp('2025-09-11 04:00:00'), 'Rain': 0.08770658302013315, 'Wind': 11.179603450006182, 'Temp': 11.897302841056558}, {'Date': Timestamp('2025-09-12 04:00:00'), 'Rain': 0.10348069642920942, 'Wind': 11.188766344587146, 'Temp': 13.361540803099047}, {'Date': Timestamp('2025-09-13 04:00:00'), 'Rain': 0.0972258329862283, 'Wind': 11.060772208468052, 'Temp': 12.563499574513912}, {'Date': Timestamp('2025-09-14 04:00:00'), 'Rain': 0.10141099688946395, 'Wind': 11.205167762424436,

In [ ]:
# ## Example 
# date1 = '2023-01-01'
# date2 = '2023-01-03'
# date_list = split_dates(date1, date2)

# pred_list = []
# for each in date_list:
#         prediction = get_prediction(each, 50, -80)
#         pred_list.append(prediction)

# print(pred_list)

[{'Date': Timestamp('2023-01-01 00:00:00'), 'Rain': 1.861835249021603e-05, 'Wind': 99796.17080719717, 'Temp': 263.85566124802926}, {'Date': Timestamp('2023-01-02 00:00:00'), 'Rain': 1.872330219914592e-05, 'Wind': 99795.55069591517, 'Temp': 263.8479908292444}, {'Date': Timestamp('2023-01-03 00:00:00'), 'Rain': 1.882825190807581e-05, 'Wind': 99794.93058463317, 'Temp': 263.8403204104596}]


In [ ]:
# ## Post will have 
# # Longitude/Latitude
# # Hour/Day/Month/Year
# @app.route("/request", methods=["POST"])
# def predict1():

#     data = request.get_json()

#     latitude = data['Latitude']
#     longitude = data['Longitude']
    
#     start_date = data['Start_Date']
#     end_date = data['End_Date']
#     date_list = split_dates(start_date, end_date)

#     prediction_list = []
#     #for each Date, create a dataframe, and predict the rain wind and temp
#     for each in date_list:
#         month = data['Month']
#         day = data['Day']
#         year = data['Year']
#         hour = data['Hour']


#         rain_df = {
#         'Humidity':[humidity_avg[month -1]],
#         'Pressure (Pa)': [air_pressure_avg[month -1]],
#         'Year':[year],
#         'Month':[month],
#         'Day':[day],
#         'Hour':[hour],
#         'Longitude':[longitude],
#         'Latitude':[latitude]}
    
#         wind_df = {        
#         'Surface Air Temp (K)':[0],
#         'Humidity (g/kg)': [0],
#         'Year':[year],
#         'Month':[month],
#         'Day':[day],
#         'Hour':[hour],
#         'Longitude':[longitude],
#         'Latitude':[latitude]}
   
#         temp_df = {
#         'Surface Wind Speed (m/s)':[0],
#         'Humidity (g/kg)': [0],
#         'Year':[year],
#         'Month':[month],
#         'Day':[day],
#         'Hour':[hour],
#         'Longitude':[longitude],
#         'Latitude':[latitude] }

#         rain_df1 = pd.DataFrame(rain_df)
#         wind_df = pd.DataFrame(wind_df)
#         temp_df = pd.DataFrame(temp_df)

#         rain = rain_model.predict(rain_df1)[0]
#         wind = wind_model.predict(wind_df)[0]
#         temp = temp_model.predict(temp_df)[0]

#         prediction = {'Date': each, 'Rain': rain, 'Wind':wind, 'Temp':temp}
#         prediction_list.append(prediction)
#         #end loop

#     return jsonify({ 'Predictions' : prediction_list})




In [ ]:
# date1 = '2023-01-01T08:00:00'
# date2 = '2023-01-02T08:00:00'
# dates = split_dates(date1, date2)
# # print(dates)


DatetimeIndex(['2023-01-01 08:00:00', '2023-01-02 08:00:00'], dtype='datetime64[ns]', freq='D')


In [ ]:
# pred_list = []

# for each in dates:
#         month = each.month
#         day = each.day
#         year = each.year
#         hour = each.hour


#         rain_df = {
#         'Humidity':[0],
#         'Pressure (Pa)': [0],
#         'Year':[year],
#         'Month':[month],
#         'Day':[day],
#         'Hour':[hour],
#         'Longitude':[50],
#         'Latitude':[-80]}
    
#         wind_df = {        
#         'Surface Air Temp (K)':[0],
#         'Humidity (g/kg)': [0],
#         'Year':[year],
#         'Month':[month],
#         'Day':[day],
#         'Hour':[hour],
#         'Longitude':[50],
#         'Latitude':[-80]}
   
#         temp_df = {
#         'Surface Wind Speed (m/s)':[0],
#         'Humidity (g/kg)': [0],
#         'Year':[year],
#         'Month':[month],
#         'Day':[day],
#         'Hour':[hour],
#         'Longitude':[50],
#         'Latitude':[-80] }

#         rain_df1 = pd.DataFrame(rain_df)
#         wind_df = pd.DataFrame(wind_df)
#         temp_df = pd.DataFrame(temp_df)

#         # rain = rain_model.predict(rain_df1)[0]
#         wind = wind_model.predict(wind_df)[0]
#         temp = temp_model.predict(temp_df)[0]

#         prediction = {'Date': each, 'Rain': rain, 'Wind':wind, 'Temp':temp}
#         pred_list.append(prediction)
    
# print(pred_list)

[{'Date': Timestamp('2023-01-01 08:00:00'), 'Rain': 0.001246843873389896, 'Wind': 181143.9715257681, 'Temp': 492.55329476651434}, {'Date': Timestamp('2023-01-02 08:00:00'), 'Rain': 0.001246948823098826, 'Wind': 181143.3514144861, 'Temp': 492.5456243477295}]


In [ ]:
# for each in dates:
#     print(each.day)
#     print(each.month)
#     # print(each.year)
#     print(each.hour)

1
1
2023
0
2
1
2023
0
3
1
2023
0
4
1
2023
0
5
1
2023
0
6
1
2023
0
7
1
2023
0
8
1
2023
0
9
1
2023
0
10
1
2023
0
